# Notebook 58 — Merged-Threshold Experiment

**Hypothesis**: All prior notebooks (nb53–nb57) define "high-impact" using the AUB-only
75th-percentile citation threshold, even when training on the full merged dataset
(AUB + Lehigh + Marquette + Villanova). This creates an inconsistency: the model learns
from four institutions but its labels are calibrated to one.

Replacing the AUB-only threshold with a **merged threshold** (75th percentile of citations
across all four training institutions) makes the label definition institution-agnostic and
consistent with the training distribution. This notebook tests whether that change improves
predictive performance.

**Baselines for comparison**:
- REF-CLEAN: F1=0.5109, AUC=0.8216 (TF-IDF + numerics, AUB-only threshold)
- nb54-E: F1=0.5302, AUC=0.8257 (SPECTER1 + numerics, LGBM tuned, AUB-only threshold)
- nb57-C: F1=0.5186, AUC=0.8216 (SPECTER1 + ext-numerics, LGBM tuned, AUB-only threshold, merged→AUB)

**Configs**:
- **Config A**: SPECTER1 + raw numerics, LGBM tuned, merged threshold, merged→AUB
- **Config B**: SPECTER1 + raw numerics, LGBM tuned, AUB threshold, merged→AUB  *(direct comparison)*
- **Config C**: SPECTER1 + raw numerics, LGBM tuned, merged threshold, merged→all institutions
- **Config D**: Numeric-only ablation, LGBM tuned, merged threshold, merged→AUB

In [1]:
import sys
sys.path.append('../../')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pickle
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, cohen_kappa_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
TRAIN_YEARS  = list(range(2010, 2018))
TEST_YEARS   = [2018, 2019, 2020]
QUANTILE     = 0.75

NB54_REF_CLEAN_F1  = 0.5109
NB54_REF_CLEAN_AUC = 0.8216
NB54_BEST_F1       = 0.5302
NB54_BEST_AUC      = 0.8257
NB57_C_F1          = 0.5186
NB57_C_AUC         = 0.8216

CACHE_DIR = Path('../../data/cache')
print('Libraries loaded')

Libraries loaded


## 1. Load data & splits

In [2]:
data_path = Path('../../data/processed/all_unis_cleaned.pkl')
df = pd.read_pickle(data_path)

df_train    = df[df['Year'].isin(TRAIN_YEARS)].copy()
df_test_all = df[df['Year'].isin(TEST_YEARS)].copy()
df_test_aub = df_test_all[df_test_all['institution'] == 'AUB'].copy()
df_aub_train = df_train[df_train['institution'] == 'AUB'].copy()

# --- Two threshold definitions ---
thr_aub    = df_aub_train['Citations'].quantile(QUANTILE)
thr_merged = df_train['Citations'].quantile(QUANTILE)

print(f"AUB-only threshold (prior notebooks): {thr_aub:.0f} citations")
print(f"Merged threshold   (this notebook):   {thr_merged:.0f} citations")
print(f"Difference: {thr_merged - thr_aub:+.0f} citations")
print()

# Labels under each threshold
y_train_aub_thr    = (df_train['Citations'] >= thr_aub).astype(int)
y_train_merged_thr = (df_train['Citations'] >= thr_merged).astype(int)
y_test_aub_aub_thr    = (df_test_aub['Citations'] >= thr_aub).astype(int)
y_test_aub_merged_thr = (df_test_aub['Citations'] >= thr_merged).astype(int)
y_test_all_merged_thr = (df_test_all['Citations'] >= thr_merged).astype(int)

print(f"Train positive rate — AUB threshold:    {y_train_aub_thr.mean():.1%}")
print(f"Train positive rate — merged threshold: {y_train_merged_thr.mean():.1%}")
print(f"AUB test positive rate — AUB threshold:    {y_test_aub_aub_thr.mean():.1%}")
print(f"AUB test positive rate — merged threshold: {y_test_aub_merged_thr.mean():.1%}")
print(f"\nMerged train: {len(df_train):,}  |  AUB test: {len(df_test_aub):,}  |  All-inst test: {len(df_test_all):,}")

# Per-institution breakdown
print("\nPer-institution positive rates (merged threshold, train):")
for inst, grp in df_train.groupby('institution'):
    pos = (grp['Citations'] >= thr_merged).mean()
    print(f"  {inst:20s}: {pos:.1%}  (n={len(grp):,})")

AUB-only threshold (prior notebooks): 40 citations
Merged threshold   (this notebook):   43 citations
Difference: +3 citations

Train positive rate — AUB threshold:    27.2%
Train positive rate — merged threshold: 25.2%
AUB test positive rate — AUB threshold:    18.0%
AUB test positive rate — merged threshold: 16.5%

Merged train: 15,391  |  AUB test: 3,573  |  All-inst test: 8,871

Per-institution positive rates (merged threshold, train):
  AUB                 : 23.5%  (n=5,569)
  Lehigh              : 30.0%  (n=3,918)
  Marquette           : 22.8%  (n=3,342)
  Villanova           : 24.9%  (n=2,562)


## 2. Feature matrices

In [3]:
COL_MAP = {
    'snip':             'SNIP (publication year)',
    'snip_pct':         'SNIP percentile',
    'citescore':        'CiteScore (publication year)',
    'citescore_pct':    'CiteScore percentile',
    'sjr':              'SJR (publication year)',
    'sjr_pct':          'SJR percentile',
    'topic_prom':       'Topic Prominence Percentile',
    'num_authors':      'Authors',
    'num_institutions': 'Affiliations',
    'num_countries':    'Countries',
}


def extract_numeric(subset_df):
    vf = pd.DataFrame(index=subset_df.index)
    for feat, col in COL_MAP.items():
        if col in subset_df.columns:
            if col in ('Authors', 'Affiliations', 'Countries'):
                vf[feat] = subset_df[col].fillna('').apply(
                    lambda x: len(str(x).split(';')) if x else 1
                )
            else:
                vf[feat] = pd.to_numeric(subset_df[col], errors='coerce')
    return vf


raw_tr = extract_numeric(df_train)
raw_te_aub = extract_numeric(df_test_aub)
raw_te_all = extract_numeric(df_test_all)

med = raw_tr.median()
sc  = StandardScaler()

X_num_tr     = pd.DataFrame(sc.fit_transform(raw_tr.fillna(med)),         index=raw_tr.index,     columns=raw_tr.columns)
X_num_te_aub = pd.DataFrame(sc.transform(raw_te_aub.fillna(med)),         index=raw_te_aub.index, columns=raw_tr.columns)
X_num_te_all = pd.DataFrame(sc.transform(raw_te_all.fillna(med)),         index=raw_te_all.index, columns=raw_tr.columns)

print(f"Numeric features: {X_num_tr.shape[1]}")

# Load SPECTER1 embeddings
SPECTER1_CACHE = CACHE_DIR / 'specter_embeddings_merged.pkl'
X_sp_tr = X_sp_te_aub = X_sp_te_all = None

if SPECTER1_CACHE.exists():
    with open(SPECTER1_CACHE, 'rb') as f:
        cache = pickle.load(f)
    emb_tr      = cache['train_merged']
    emb_te_aub  = cache['test_aub']

    specter_cols = [f'sp_{i}' for i in range(emb_tr.shape[1])]
    sp_sc = StandardScaler()
    sp_tr_scaled     = pd.DataFrame(sp_sc.fit_transform(emb_tr),     index=df_train.index,    columns=specter_cols)
    sp_te_aub_scaled = pd.DataFrame(sp_sc.transform(emb_te_aub),     index=df_test_aub.index, columns=specter_cols)

    X_sp_tr     = pd.concat([sp_tr_scaled,     X_num_tr.set_index(sp_tr_scaled.index)],     axis=1)
    X_sp_te_aub = pd.concat([sp_te_aub_scaled, X_num_te_aub.set_index(sp_te_aub_scaled.index)], axis=1)

    # For all-institution test, check if cache has full test embeddings
    if 'test_all' in cache:
        emb_te_all = cache['test_all']
        sp_te_all_scaled = pd.DataFrame(sp_sc.transform(emb_te_all), index=df_test_all.index, columns=specter_cols)
        X_sp_te_all = pd.concat([sp_te_all_scaled, X_num_te_all.set_index(sp_te_all_scaled.index)], axis=1)

    print(f"SPECTER1 + numeric: train {X_sp_tr.shape}  AUB test {X_sp_te_aub.shape}")
else:
    print('SPECTER1 cache not found — SPECTER configs will be skipped. Run nb55 first.')

Numeric features: 5
SPECTER1 + numeric: train (15391, 773)  AUB test (3573, 773)


## 3. Evaluation helper

In [4]:
PARAM_DIST = {
    'n_estimators':      [200, 300, 500],
    'learning_rate':     [0.01, 0.03, 0.05, 0.1],
    'num_leaves':        [15, 31, 63, 127],
    'min_child_samples': [10, 20, 30, 50],
    'subsample':         [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree':  [0.6, 0.7, 0.8, 0.9],
    'reg_alpha':         [0.0, 0.1, 0.5, 1.0],
    'reg_lambda':        [0.0, 0.1, 0.5, 1.0],
}


def evaluate(X_tr, y_tr, X_te, y_te, label):
    n_pos = int(y_tr.sum())
    n_neg = int((y_tr == 0).sum())
    scale_w = n_neg / n_pos

    base = LGBMClassifier(scale_pos_weight=scale_w, random_state=RANDOM_STATE, verbose=-1)
    cv   = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        base, PARAM_DIST, n_iter=25, scoring='f1',
        cv=cv, random_state=RANDOM_STATE, n_jobs=-1, verbose=0
    )
    search.fit(X_tr, y_tr)
    model  = search.best_estimator_
    cv_f1  = search.best_score_

    proba = model.predict_proba(X_te)[:, 1]
    thresholds = np.arange(0.10, 0.91, 0.01)
    f1s    = [f1_score(y_te, (proba >= t).astype(int), zero_division=0) for t in thresholds]
    best_t = thresholds[int(np.argmax(f1s))]
    y_pred = (proba >= best_t).astype(int)

    return {
        'label':     label,
        'f1':        f1_score(y_te, y_pred, zero_division=0),
        'auc':       roc_auc_score(y_te, proba),
        'kappa':     cohen_kappa_score(y_te, y_pred),
        'recall':    recall_score(y_te, y_pred, zero_division=0),
        'precision': precision_score(y_te, y_pred, zero_division=0),
        'threshold': best_t,
        'cv_f1':     cv_f1,
        'model':     model,
    }


def print_result(r):
    df1  = r['f1']  - NB54_REF_CLEAN_F1
    dauc = r['auc'] - NB54_REF_CLEAN_AUC
    print(f"  F1:        {r['f1']:.4f}  ({df1:+.4f} vs REF-CLEAN)")
    print(f"  AUC:       {r['auc']:.4f}  ({dauc:+.4f} vs REF-CLEAN)")
    print(f"  Kappa:     {r['kappa']:.4f}")
    print(f"  Recall:    {r['recall']:.4f}  |  Precision: {r['precision']:.4f}  |  Threshold: {r['threshold']:.2f}")
    print(f"  CV F1:     {r['cv_f1']:.4f}")


print('Evaluation helper ready.')

Evaluation helper ready.


## 4. Experiments

### Config A — SPECTER1 + numerics, **merged threshold**, merged→AUB

In [5]:
results = []

print('=' * 70)
print('Config A: SPECTER1 + numerics, merged threshold, merged→AUB')
print('=' * 70)

if X_sp_tr is None:
    print('Skipped — SPECTER1 cache not found.')
    res_a = None
else:
    res_a = evaluate(
        X_sp_tr.values, y_train_merged_thr,
        X_sp_te_aub.values, y_test_aub_merged_thr,
        label='Config A (SPECTER1+num, merged-thr, merged→AUB)',
    )
    results.append(res_a)
    print_result(res_a)

Config A: SPECTER1 + numerics, merged threshold, merged→AUB
  F1:        0.5075  (-0.0034 vs REF-CLEAN)
  AUC:       0.8238  (+0.0022 vs REF-CLEAN)
  Kappa:     0.3917
  Recall:    0.6020  |  Precision: 0.4387  |  Threshold: 0.62
  CV F1:     0.5779


### Config B — SPECTER1 + numerics, **AUB threshold**, merged→AUB  *(direct comparison)*

In [6]:
print('=' * 70)
print('Config B: SPECTER1 + numerics, AUB threshold, merged→AUB (baseline)')
print('=' * 70)

if X_sp_tr is None:
    print('Skipped — SPECTER1 cache not found.')
    res_b = None
else:
    res_b = evaluate(
        X_sp_tr.values, y_train_aub_thr,
        X_sp_te_aub.values, y_test_aub_aub_thr,
        label='Config B (SPECTER1+num, AUB-thr, merged→AUB)',
    )
    results.append(res_b)
    print_result(res_b)

Config B: SPECTER1 + numerics, AUB threshold, merged→AUB (baseline)
  F1:        0.5183  (+0.0074 vs REF-CLEAN)
  AUC:       0.8195  (-0.0021 vs REF-CLEAN)
  Kappa:     0.3894
  Recall:    0.6267  |  Precision: 0.4419  |  Threshold: 0.60
  CV F1:     0.5927


### Config C — SPECTER1 + numerics, merged threshold, merged→all institutions

In [7]:
print('=' * 70)
print('Config C: SPECTER1 + numerics, merged threshold, merged→all institutions')
print('=' * 70)

if X_sp_te_all is None:
    print('Skipped — full test embeddings not in cache (test_all key missing). Run nb55 with full test set.')
    res_c = None
else:
    res_c = evaluate(
        X_sp_tr.values, y_train_merged_thr,
        X_sp_te_all.values, y_test_all_merged_thr,
        label='Config C (SPECTER1+num, merged-thr, merged→all-inst)',
    )
    results.append(res_c)
    print_result(res_c)

    # Per-institution breakdown
    print('\nPer-institution breakdown:')
    model_c = res_c['model']
    proba_all = model_c.predict_proba(X_sp_te_all.values)[:, 1]
    for inst, grp in df_test_all.groupby('institution'):
        mask = df_test_all['institution'] == inst
        p    = proba_all[mask.values]
        y_i  = y_test_all_merged_thr[mask]
        if y_i.sum() == 0:
            print(f"  {inst:20s}: no positives in test set")
            continue
        thresholds = np.arange(0.10, 0.91, 0.01)
        f1s    = [f1_score(y_i, (p >= t).astype(int), zero_division=0) for t in thresholds]
        best_t = thresholds[int(np.argmax(f1s))]
        y_pred_i = (p >= best_t).astype(int)
        print(f"  {inst:20s}: F1={f1_score(y_i, y_pred_i):.4f}  AUC={roc_auc_score(y_i, p):.4f}  n={len(y_i)}")

Config C: SPECTER1 + numerics, merged threshold, merged→all institutions
  F1:        0.5099  (-0.0010 vs REF-CLEAN)
  AUC:       0.8179  (-0.0037 vs REF-CLEAN)
  Kappa:     0.3914
  Recall:    0.5995  |  Precision: 0.4436  |  Threshold: 0.64
  CV F1:     0.5779

Per-institution breakdown:
  AUB                 : F1=0.5075  AUC=0.8238  n=3573
  Lehigh              : F1=0.5341  AUC=0.8009  n=2107
  Marquette           : F1=0.5476  AUC=0.8296  n=1870
  Villanova           : F1=0.4842  AUC=0.8147  n=1321


### Config D — Numeric only, merged threshold, merged→AUB  *(ablation)*

In [8]:
print('=' * 70)
print('Config D: Numeric only, merged threshold, merged→AUB (ablation)')
print('=' * 70)

res_d = evaluate(
    X_num_tr.values, y_train_merged_thr,
    X_num_te_aub.values, y_test_aub_merged_thr,
    label='Config D (numeric only, merged-thr, merged→AUB)',
)
results.append(res_d)
print_result(res_d)

Config D: Numeric only, merged threshold, merged→AUB (ablation)
  F1:        0.4689  (-0.0420 vs REF-CLEAN)
  AUC:       0.7905  (-0.0311 vs REF-CLEAN)
  Kappa:     0.3353
  Recall:    0.6020  |  Precision: 0.3839  |  Threshold: 0.60
  CV F1:     0.5427


## 5. Results summary

In [9]:
ref_rows = [
    {'label': 'REF-CLEAN (TF-IDF+num, LR, AUB-thr, AUB→AUB)',
     'f1': NB54_REF_CLEAN_F1, 'auc': NB54_REF_CLEAN_AUC, 'kappa': None},
    {'label': 'nb54-E (SPECTER1+num, LGBM tuned, AUB-thr, AUB→AUB)',
     'f1': NB54_BEST_F1, 'auc': NB54_BEST_AUC, 'kappa': None},
    {'label': 'nb57-C (SPECTER1+ext-num, LGBM tuned, AUB-thr, merged→AUB)',
     'f1': NB57_C_F1, 'auc': NB57_C_AUC, 'kappa': None},
]

res_df = pd.DataFrame(ref_rows + [{
    'label': r['label'], 'f1': r['f1'], 'auc': r['auc'], 'kappa': r['kappa']
} for r in results if r is not None])

res_df['delta_f1']  = res_df['f1']  - NB54_REF_CLEAN_F1
res_df['delta_auc'] = res_df['auc'] - NB54_REF_CLEAN_AUC

print('\n' + '=' * 120)
print('RESULTS SUMMARY — Notebook 58: Merged Threshold Experiment')
print('=' * 120)
cols = ['label', 'f1', 'delta_f1', 'auc', 'delta_auc', 'kappa']
print(res_df[cols].to_string(index=False, float_format='{:.4f}'.format))

# Key comparison: Config A (merged-thr) vs Config B (AUB-thr)
r_a = next((r for r in results if r and 'merged-thr' in r['label'] and 'AUB' in r['label'] and 'all' not in r['label']), None)
r_b = next((r for r in results if r and 'AUB-thr' in r['label']), None)
if r_a and r_b:
    delta = r_a['f1'] - r_b['f1']
    verdict = 'HELPED' if delta > 0.005 else ('HURT' if delta < -0.005 else 'FLAT')
    print(f"\n--- Threshold impact (A vs B, same features/training) ---")
    print(f"Merged threshold vs AUB threshold:  ΔF1={delta:+.4f}  ΔAUC={r_a['auc']-r_b['auc']:+.4f}  → {verdict}")

best = res_df.loc[res_df['f1'].idxmax()]
print(f"\nBest config: {best['label']}")
print(f"  F1:  {best['f1']:.4f}  ({best['delta_f1']:+.4f} vs REF-CLEAN)")
print(f"  Gap to supervisor target (0.75): {0.75 - best['f1']:+.4f}")


RESULTS SUMMARY — Notebook 58: Merged Threshold Experiment
                                                     label     f1  delta_f1    auc  delta_auc  kappa
              REF-CLEAN (TF-IDF+num, LR, AUB-thr, AUB→AUB) 0.5109    0.0000 0.8216     0.0000    NaN
       nb54-E (SPECTER1+num, LGBM tuned, AUB-thr, AUB→AUB) 0.5302    0.0193 0.8257     0.0041    NaN
nb57-C (SPECTER1+ext-num, LGBM tuned, AUB-thr, merged→AUB) 0.5186    0.0077 0.8216     0.0000    NaN
           Config A (SPECTER1+num, merged-thr, merged→AUB) 0.5075   -0.0034 0.8238     0.0022 0.3917
              Config B (SPECTER1+num, AUB-thr, merged→AUB) 0.5183    0.0074 0.8195    -0.0021 0.3894
      Config C (SPECTER1+num, merged-thr, merged→all-inst) 0.5099   -0.0010 0.8179    -0.0037 0.3914
           Config D (numeric only, merged-thr, merged→AUB) 0.4689   -0.0420 0.7905    -0.0311 0.3353

--- Threshold impact (A vs B, same features/training) ---
Merged threshold vs AUB threshold:  ΔF1=-0.0108  ΔAUC=+0.0043  → HURT

Be

## 6. Conclusions

In [10]:
print('=' * 70)
print('NOTEBOOK 58 — CONCLUSIONS')
print('=' * 70)

for row in ref_rows:
    d = row['f1'] - NB54_REF_CLEAN_F1
    print(f"  REFERENCE  {row['label']:65s}  F1={row['f1']:.4f} ({d:+.4f})  AUC={row['auc']:.4f}")

for r in results:
    if r is None:
        continue
    df1  = r['f1']  - NB54_REF_CLEAN_F1
    dauc = r['auc'] - NB54_REF_CLEAN_AUC
    outcome = 'IMPROVED' if df1 > 0.01 else ('DEGRADED' if df1 < -0.01 else 'FLAT')
    print(f"  {outcome:9s}  {r['label']:65s}  F1={r['f1']:.4f} ({df1:+.4f})  AUC={r['auc']:.4f} ({dauc:+.4f})  Kappa={r['kappa']:.4f}")

r_a = next((r for r in results if r and 'merged-thr' in r['label'] and 'all' not in r['label']), None)
r_b = next((r for r in results if r and 'AUB-thr' in r['label']), None)
if r_a and r_b:
    delta = r_a['f1'] - r_b['f1']
    verdict = 'HELPED' if delta > 0.005 else ('HURT' if delta < -0.005 else 'FLAT')
    print(f"\nUsing the merged threshold instead of AUB-only threshold: ΔF1={delta:+.4f} → {verdict}")
    if abs(delta) <= 0.005:
        print("  → Label definition does not significantly affect performance.")
        print("  → AUB citation distribution is representative of the merged pool at this quantile.")
    elif delta > 0.005:
        print("  → Merged threshold improves label consistency and downstream F1.")
        print("  → Prior notebooks' AUB-only threshold was suboptimal for merged training.")
    else:
        print("  → AUB-only threshold was better calibrated for AUB test performance.")
        print("  → Merged threshold inflates/deflates positives in a way that hurts AUB generalisation.")

NOTEBOOK 58 — CONCLUSIONS
  REFERENCE  REF-CLEAN (TF-IDF+num, LR, AUB-thr, AUB→AUB)                       F1=0.5109 (+0.0000)  AUC=0.8216
  REFERENCE  nb54-E (SPECTER1+num, LGBM tuned, AUB-thr, AUB→AUB)                F1=0.5302 (+0.0193)  AUC=0.8257
  REFERENCE  nb57-C (SPECTER1+ext-num, LGBM tuned, AUB-thr, merged→AUB)         F1=0.5186 (+0.0077)  AUC=0.8216
  FLAT       Config A (SPECTER1+num, merged-thr, merged→AUB)                    F1=0.5075 (-0.0034)  AUC=0.8238 (+0.0022)  Kappa=0.3917
  FLAT       Config B (SPECTER1+num, AUB-thr, merged→AUB)                       F1=0.5183 (+0.0074)  AUC=0.8195 (-0.0021)  Kappa=0.3894
  FLAT       Config C (SPECTER1+num, merged-thr, merged→all-inst)               F1=0.5099 (-0.0010)  AUC=0.8179 (-0.0037)  Kappa=0.3914
  DEGRADED   Config D (numeric only, merged-thr, merged→AUB)                    F1=0.4689 (-0.0420)  AUC=0.7905 (-0.0311)  Kappa=0.3353

Using the merged threshold instead of AUB-only threshold: ΔF1=-0.0108 → HURT
  → AUB-only thr